# Energy Demand Forecasting — Model Training & Comparison

This notebook trains and evaluates four forecasting approaches:

| Model | Type | Description |
|-------|------|-------------|
| **Seasonal Naive** | Baseline | Same hour from previous week |
| **Prophet** | Statistical | Additive model with seasonality decomposition |
| **LSTM** | Deep Learning | 2-layer stacked LSTM (PyTorch) |
| **TCN** | Deep Learning | Dilated causal convolutions (PyTorch) |

**Forecast task**: Given 168 hours (1 week) of history, predict the next 24 hours.  
**Metrics**: MAE (MW), RMSE (MW), MAPE (%)

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json

from src.train import prepare_data, create_dataloaders, train_model, EnergyDataset
from src.data.features import FEATURE_COLUMNS
from src.models.lstm import EnergyLSTM
from src.models.tcn import EnergyTCN
from src.evaluate import (
    evaluate_pytorch_model,
    evaluate_seasonal_naive,
    print_comparison_table,
    plot_predictions,
    plot_training_history,
)

device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using device: {device}")

%matplotlib inline

In [ ]:
import sys
sys.path.insert(0, "..")

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json

from src.train import prepare_data, create_dataloaders, train_model, EnergyDataset
from src.data.features import FEATURE_COLUMNS
from src.models.lstm import EnergyLSTM
from src.models.tcn import EnergyTCN
from src.evaluate import (
    evaluate_pytorch_model,
    evaluate_seasonal_naive,
    print_comparison_table,
    plot_predictions,
    plot_training_history,
)

device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using device: {device}")

%matplotlib inline

# Full data pipeline: load -> split -> scale -> feature engineering
train_df, val_df, test_df, scaler = prepare_data("data/PJME_hourly.csv")

In [ ]:
# Full data pipeline: load -> split -> scale -> feature engineering
train_df, val_df, test_df, scaler = prepare_data("../data/PJME_hourly.csv")

In [ ]:
# Hyperparameters
WINDOW_SIZE = 168        # 1 week of hourly data
FORECAST_HORIZON = 24   # predict next 24 hours
BATCH_SIZE = 64
EPOCHS = 50
LR = 1e-3

INPUT_SIZE = len(FEATURE_COLUMNS)
print(f"Input features: {INPUT_SIZE}")

train_loader, val_loader, test_loader = create_dataloaders(
    train_df, val_df, test_df,
    feature_cols=FEATURE_COLUMNS,
    window_size=WINDOW_SIZE,
    forecast_horizon=FORECAST_HORIZON,
    batch_size=BATCH_SIZE,
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")

lstm_model = EnergyLSTM(
    input_size=INPUT_SIZE,
    hidden_size=128,
    num_layers=2,
    dropout=0.2,
    forecast_horizon=FORECAST_HORIZON,
)

print(f"LSTM parameters: {sum(p.numel() for p in lstm_model.parameters()):,}")

lstm_model, lstm_history = train_model(
    lstm_model, train_loader, val_loader,
    epochs=EPOCHS, lr=LR, model_name="lstm",
    save_dir="models",
)

In [ ]:
plot_training_history(lstm_history, "LSTM", save_path="models/lstm_training.png")

In [ ]:
plot_training_history(lstm_history, "LSTM", save_path="../models/lstm_training.png")

tcn_model = EnergyTCN(
    input_size=INPUT_SIZE,
    num_channels=[64, 64, 64, 64],
    kernel_size=3,
    dropout=0.2,
    forecast_horizon=FORECAST_HORIZON,
)

print(f"TCN parameters: {sum(p.numel() for p in tcn_model.parameters()):,}")

tcn_model, tcn_history = train_model(
    tcn_model, train_loader, val_loader,
    epochs=EPOCHS, lr=LR, model_name="tcn",
    save_dir="models",
)

In [ ]:
plot_training_history(tcn_history, "TCN", save_path="models/tcn_training.png")

In [ ]:
plot_training_history(tcn_history, "TCN", save_path="../models/tcn_training.png")

# Seasonal naive uses the raw MW values (no scaling needed)
from src.data.preprocess import load_data, temporal_split

df_raw = load_data("data/PJME_hourly.csv")
_, _, test_raw = temporal_split(df_raw)
test_mw = test_raw["consumption_mw"].values

naive_results = evaluate_seasonal_naive(test_mw, season_length=168, horizon=24)
print(f"Seasonal Naive — MAE: {naive_results['mae']:.2f} MW | RMSE: {naive_results['rmse']:.2f} MW | MAPE: {naive_results['mape']:.2f}%")

In [ ]:
# Seasonal naive uses the raw MW values (no scaling needed)
from src.data.preprocess import load_data, temporal_split

df_raw = load_data("../data/PJME_hourly.csv")
_, _, test_raw = temporal_split(df_raw)
test_mw = test_raw["consumption_mw"].values

naive_results = evaluate_seasonal_naive(test_mw, season_length=168, horizon=24)
print(f"Seasonal Naive — MAE: {naive_results['mae']:.2f} MW | RMSE: {naive_results['rmse']:.2f} MW | MAPE: {naive_results['mape']:.2f}%")

## 5. Baseline: Prophet

In [ ]:
from src.models.baselines import ProphetBaseline
from src.evaluate import mae, rmse, mape

# Train Prophet on training data
train_raw, _, test_raw = temporal_split(df_raw)

prophet_model = ProphetBaseline()
prophet_model.fit(train_raw)

# Predict over the full test period
test_hours = len(test_raw)
prophet_forecast = prophet_model.predict(periods=test_hours, freq="h")

# Align predictions with actuals
prophet_preds = prophet_forecast["yhat"].values
prophet_actuals = test_raw["consumption_mw"].values[:len(prophet_preds)]

prophet_results = {
    "mae": mae(prophet_actuals, prophet_preds),
    "rmse": rmse(prophet_actuals, prophet_preds),
    "mape": mape(prophet_actuals, prophet_preds),
    "predictions": prophet_preds,
    "actuals": prophet_actuals,
}

print(f"Prophet — MAE: {prophet_results['mae']:.2f} MW | RMSE: {prophet_results['rmse']:.2f} MW | MAPE: {prophet_results['mape']:.2f}%")

## 6. Evaluate Deep Learning Models on Test Set

In [ ]:
lstm_results = evaluate_pytorch_model(lstm_model, test_loader, scaler, device)
tcn_results = evaluate_pytorch_model(tcn_model, test_loader, scaler, device)

print(f"LSTM — MAE: {lstm_results['mae']:.2f} MW | RMSE: {lstm_results['rmse']:.2f} MW | MAPE: {lstm_results['mape']:.2f}%")
print(f"TCN  — MAE: {tcn_results['mae']:.2f} MW | RMSE: {tcn_results['rmse']:.2f} MW | MAPE: {tcn_results['mape']:.2f}%")

## 7. Model Comparison

In [ ]:
# Bar chart comparison
model_names = list(all_results.keys())
mae_vals = [all_results[m]["mae"] for m in model_names]
rmse_vals = [all_results[m]["rmse"] for m in model_names]

x = np.arange(len(model_names))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, mae_vals, width, label="MAE (MW)", color="#2196F3")
bars2 = ax.bar(x + width/2, rmse_vals, width, label="RMSE (MW)", color="#FF5722")

ax.set_ylabel("Error (MW)")
ax.set_title("Model Comparison — Test Set Performance")
ax.set_xticks(x)
ax.set_xticklabels(model_names)
ax.legend()
ax.grid(True, alpha=0.3, axis="y")

# Add value labels on bars
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 20,
            f"{bar.get_height():.0f}", ha="center", va="bottom", fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 20,
            f"{bar.get_height():.0f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig("models/model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Bar chart comparison
model_names = list(all_results.keys())
mae_vals = [all_results[m]["mae"] for m in model_names]
rmse_vals = [all_results[m]["rmse"] for m in model_names]

x = np.arange(len(model_names))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, mae_vals, width, label="MAE (MW)", color="#2196F3")
bars2 = ax.bar(x + width/2, rmse_vals, width, label="RMSE (MW)", color="#FF5722")

ax.set_ylabel("Error (MW)")
ax.set_title("Model Comparison — Test Set Performance")
ax.set_xticks(x)
ax.set_xticklabels(model_names)
ax.legend()
ax.grid(True, alpha=0.3, axis="y")

# Add value labels on bars
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 20,
            f"{bar.get_height():.0f}", ha="center", va="bottom", fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 20,
            f"{bar.get_height():.0f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig("../models/model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

# Plot 1 week of predictions for the deep learning models
dl_results = {"LSTM": lstm_results, "TCN": tcn_results}
plot_predictions(dl_results, sample_size=168, save_path="models/predictions_sample.png")

# Display the saved plot
from IPython.display import Image
Image(filename="models/predictions_sample.png")

In [ ]:
# Plot 1 week of predictions for the deep learning models
dl_results = {"LSTM": lstm_results, "TCN": tcn_results}
plot_predictions(dl_results, sample_size=168, save_path="../models/predictions_sample.png")

# Display the saved plot
from IPython.display import Image
Image(filename="../models/predictions_sample.png")

from src.predict import predict
from src.data.preprocess import load_data

# Use the last 2 weeks of data for a sample prediction
df_full = load_data("data/PJME_hourly.csv")
recent = df_full.tail(336)  # 2 weeks

# Get best model (use whichever performed better)
best_model = lstm_model  # or tcn_model

forecast = predict(best_model, recent, scaler)

# Display forecast
last_ts = recent.index[-1]
forecast_times = pd.date_range(start=last_ts + pd.Timedelta(hours=1), periods=24, freq="h")

forecast_df = pd.DataFrame({
    "timestamp": forecast_times,
    "predicted_mw": forecast,
})

print("24-Hour Forecast:")
print(forecast_df.to_string(index=False))

# Quick plot
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(recent.index[-48:], recent["consumption_mw"].values[-48:], label="Recent (48h)", color="#2196F3")
ax.plot(forecast_times, forecast, label="Forecast (24h)", color="#FF5722", linewidth=2, marker="o", markersize=4)
ax.axvline(last_ts, color="gray", linestyle="--", alpha=0.5, label="Forecast start")
ax.set_xlabel("Time")
ax.set_ylabel("Energy (MW)")
ax.set_title("Sample 24-Hour Forecast")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
from src.predict import predict
from src.data.preprocess import load_data

# Use the last 2 weeks of data for a sample prediction
df_full = load_data("../data/PJME_hourly.csv")
recent = df_full.tail(336)  # 2 weeks

# Get best model (use whichever performed better)
best_model = lstm_model  # or tcn_model

forecast = predict(best_model, recent, scaler)

# Display forecast
last_ts = recent.index[-1]
forecast_times = pd.date_range(start=last_ts + pd.Timedelta(hours=1), periods=24, freq="h")

forecast_df = pd.DataFrame({
    "timestamp": forecast_times,
    "predicted_mw": forecast,
})

print("24-Hour Forecast:")
print(forecast_df.to_string(index=False))

# Quick plot
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(recent.index[-48:], recent["consumption_mw"].values[-48:], label="Recent (48h)", color="#2196F3")
ax.plot(forecast_times, forecast, label="Forecast (24h)", color="#FF5722", linewidth=2, marker="o", markersize=4)
ax.axvline(last_ts, color="gray", linestyle="--", alpha=0.5, label="Forecast start")
ax.set_xlabel("Time")
ax.set_ylabel("Energy (MW)")
ax.set_title("Sample 24-Hour Forecast")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Summary

### Results

| Model | MAE (MW) | RMSE (MW) | MAPE (%) |
|-------|----------|-----------|----------|
| Seasonal Naive | — | — | — |
| Prophet | — | — | — |
| **LSTM** | — | — | — |
| **TCN** | — | — | — |

*(Fill in after training)*

### Key Takeaways
- Deep learning models (LSTM, TCN) should outperform baselines on this dataset
- TCN typically trains faster than LSTM due to parallel convolutions
- Both capture daily/weekly seasonality through the sliding window approach
- Production deployment via FastAPI endpoint is ready in `api/app.py`

### Next Steps
- Add weather covariates (temperature from Open-Meteo API)
- Experiment with Temporal Fusion Transformer (TFT)
- Set up automated retraining pipeline
- Add data drift monitoring in production